In [1]:
from pathlib import Path

import cv2

INPUT_DIR = Path(r"D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\Gambar Final")
OUTPUT_DIR = Path(r"D:\00 Kuliah\TA\Experiment\Model\Analysis of Microalgal Density Estimation by Using LASSO and Image Texture Features\cropped_roi_50x50")

EXPECTED_WIDTH = 200
EXPECTED_HEIGHT = 200
ROI_X = 68  # koordinat kiri x
ROI_Y = 68  # koordinat atas y
ROI_WIDTH = 50
ROI_HEIGHT = 50

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}


def crop_roi(image):
    """Return ROI [y: y+50, x: x+50], after validating source dimensions."""
    height, width = image.shape[:2]
    if (width, height) != (EXPECTED_WIDTH, EXPECTED_HEIGHT):
        raise ValueError(
            f"Ukuran gambar harus {EXPECTED_WIDTH}x{EXPECTED_HEIGHT} px, "
            f"tetapi ditemukan {width}x{height} px."
        )

    # Dalam NumPy/OpenCV urutan indeks adalah [baris, kolom] = [y, x].
    roi = image[ROI_Y : ROI_Y + ROI_HEIGHT, ROI_X : ROI_X + ROI_WIDTH]
    if roi.shape[:2] != (ROI_HEIGHT, ROI_WIDTH):
        raise ValueError(f"Ukuran ROI tidak sesuai: {roi.shape[:2]}")
    return roi


def main() -> None:
    if not INPUT_DIR.is_dir():
        raise FileNotFoundError(f"Folder gambar mentah tidak ditemukan: {INPUT_DIR}")

    image_paths = sorted(
        path for path in INPUT_DIR.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        print(f"Tidak ada gambar ditemukan di: {INPUT_DIR}")
        return

    for image_path in image_paths:
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            print(f"Lewati gambar yang tidak dapat dibaca: {image_path}")
            continue

        try:
            roi = crop_roi(image)
        except ValueError as error:
            print(f"Lewati {image_path.name}: {error}")
            continue

        # Simpan lossless sebagai PNG dan pertahankan struktur subfolder.
        relative_path = image_path.relative_to(INPUT_DIR).with_suffix(".png")
        output_path = OUTPUT_DIR / relative_path
        output_path.parent.mkdir(parents=True, exist_ok=True)
        if not cv2.imwrite(str(output_path), roi):
            print(f"Gagal menyimpan: {output_path}")
            continue

        print(
            f"OK: {image_path.name} | sumber={image.shape[1]}x{image.shape[0]} "
            f"| ROI=(x={ROI_X}, y={ROI_Y}, w={ROI_WIDTH}, h={ROI_HEIGHT}) "
            f"| hasil={roi.shape[1]}x{roi.shape[0]} | {output_path}"
        )


if __name__ == "__main__":
    main()


OK: 0001_K10_obsbot_photo_20260415_150013_1920x1080.jpg | sumber=200x200 | ROI=(x=68, y=68, w=50, h=50) | hasil=50x50 | D:\00 Kuliah\TA\Experiment\Model\Analysis of Microalgal Density Estimation by Using LASSO and Image Texture Features\cropped_roi_50x50\0001_K10_obsbot_photo_20260415_150013_1920x1080.png
OK: 0002_K10_obsbot_photo_20260416_110013_1920x1080.jpg | sumber=200x200 | ROI=(x=68, y=68, w=50, h=50) | hasil=50x50 | D:\00 Kuliah\TA\Experiment\Model\Analysis of Microalgal Density Estimation by Using LASSO and Image Texture Features\cropped_roi_50x50\0002_K10_obsbot_photo_20260416_110013_1920x1080.png
OK: 0003_K10_obsbot_photo_20260417_115513_1920x1080.jpg | sumber=200x200 | ROI=(x=68, y=68, w=50, h=50) | hasil=50x50 | D:\00 Kuliah\TA\Experiment\Model\Analysis of Microalgal Density Estimation by Using LASSO and Image Texture Features\cropped_roi_50x50\0003_K10_obsbot_photo_20260417_115513_1920x1080.png
OK: 0004_K10_obsbot_photo_20260418_115513_1920x1080.jpg | sumber=200x200 | ROI=